# 農地問題エリア検出システム

**画像ソース**: 国土地理院 全国最新写真（シームレス） - APIキー不要・完全無料

## 実行順序
1. セル1〜3: 初期セットアップ（最初に一度だけ）
2. セル4: 設定（GeoJSONパスを変更）
3. セル5〜9: 画像取得
4. **手動作業**: `data/unlabeled/` の画像を `data/farmland/` と `data/problem/` に振り分け
5. セル10〜13: CNN学習
6. セル14〜16: 推論・自動仕分け
7. **手動作業**: `data/review/` の画像を振り分け
8. セル17: 継続学習（精度が上がるまで 手動仕分け→継続学習 を繰り返す）

In [ ]:
# ===== セル1: ライブラリインストール（初回のみ） =====
!pip install geopandas shapely Pillow torch torchvision tqdm matplotlib requests scikit-learn pandas pyproj ipywidgets -q
# tqdmのプログレスバーをJupyterで表示するために ipywidgets が必要
# インストール後にカーネルを再起動してください（メニュー: Kernel → Restart）

In [ ]:
# ===== セル2: インポート =====
import hashlib
import io
import math
import shutil
import time
import warnings
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
from PIL import Image, ImageDraw
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, models

# tqdm: notebook環境で固まる場合は tqdm.auto が自動判別して安全
try:
    from tqdm.auto import tqdm
except ImportError:
    from tqdm import tqdm

warnings.filterwarnings('ignore')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'デバイス: {DEVICE}')

In [ ]:
# ===== セル3: ディレクトリ作成 =====
for d in ['data/unlabeled', 'data/farmland', 'data/problem', 'data/review', 'models', 'logs']:
    Path(d).mkdir(parents=True, exist_ok=True)
print('ディレクトリ作成完了')

In [ ]:
# ===== セル4: 設定（ここを自分の環境に合わせて変更） =====
GEOJSON_PATH = 'data/farmland.geojson'  # ← 農地GeoJSONのパスを指定
OUTPUT_DIR   = 'data/unlabeled'
MAX_POLYGONS = 200    # None で全件。まず小さい数でテスト推奨
ZOOM         = 18     # 18=約0.6m/pixel。大きいポリゴンは自動で下げる
OUT_SIZE     = 512    # 保存する画像サイズ（px）
MARGIN_TILES = 1      # ポリゴン周囲の余白タイル数

# 国土地理院タイルURL（APIキー不要・無料）
GSI_TILE_URL = 'https://cyberjapandata.gsi.go.jp/xyz/seamlessphoto/{z}/{x}/{y}.jpg'

In [ ]:
# ===== セル5: 画像取得ヘルパー関数 =====
TILE_SIZE = 256  # GSIタイルは256x256px固定

def latlon_to_tile_xy(lat, lon, zoom):
    tx = int((lon + 180) / 360 * 2**zoom)
    sin_lat = math.sin(math.radians(lat))
    ty = int((1 - math.log((1 + sin_lat) / (1 - sin_lat)) / (4 * math.pi)) / 2 * 2**zoom)
    return tx, ty

def latlon_to_pixel_in_canvas(lat, lon, zoom, origin_tx, origin_ty):
    x_frac = (lon + 180) / 360 * 2**zoom
    sin_lat = math.sin(math.radians(lat))
    y_frac = (1 - math.log((1 + sin_lat) / (1 - sin_lat)) / (4 * math.pi)) / 2 * 2**zoom
    return int((x_frac - origin_tx) * TILE_SIZE), int((y_frac - origin_ty) * TILE_SIZE)

def fetch_tile(tx, ty, zoom, session, retries=3):
    """1枚のXYZタイルを取得。404はNoneを返す（フォールバック用）。失敗はグレー。"""
    url = GSI_TILE_URL.format(z=zoom, x=tx, y=ty)
    for attempt in range(retries):
        try:
            resp = session.get(url, timeout=15)
            if resp.status_code == 404:
                return None  # このzoomでタイルが存在しない
            resp.raise_for_status()
            return Image.open(io.BytesIO(resp.content)).convert('RGB')
        except Exception:
            if attempt < retries - 1:
                time.sleep(1.5 ** attempt)
    return Image.new('RGB', (TILE_SIZE, TILE_SIZE), (128, 128, 128))

def find_available_zoom(lat, lon, max_zoom=18, min_zoom=10, session=None):
    """
    指定座標でタイルが存在する最大ズームレベルを返す。
    max_zoom から下に向かって404でなくなるzoomを探す。
    """
    if session is None:
        session = requests.Session()
    for z in range(max_zoom, min_zoom - 1, -1):
        tx, ty = latlon_to_tile_xy(lat, lon, z)
        url = GSI_TILE_URL.format(z=z, x=tx, y=ty)
        try:
            resp = session.get(url, timeout=10)
            if resp.status_code == 200:
                return z
        except Exception:
            pass
    return min_zoom

def fetch_region_image(bounds, zoom, margin=1, session=None):
    """タイルを取得してキャンバスに結合・クロップして返す。"""
    minx, miny, maxx, maxy = bounds
    if session is None:
        session = requests.Session()
    tx_min, ty_min = latlon_to_tile_xy(maxy, minx, zoom)
    tx_max, ty_max = latlon_to_tile_xy(miny, maxx, zoom)
    tx_min -= margin; ty_min -= margin
    tx_max += margin; ty_max += margin
    n_cols = tx_max - tx_min + 1
    n_rows = ty_max - ty_min + 1
    canvas = Image.new('RGB', (n_cols * TILE_SIZE, n_rows * TILE_SIZE))
    for row, ty in enumerate(range(ty_min, ty_max + 1)):
        for col, tx in enumerate(range(tx_min, tx_max + 1)):
            tile = fetch_tile(tx, ty, zoom, session)
            if tile:
                canvas.paste(tile, (col * TILE_SIZE, row * TILE_SIZE))
    left,  top    = latlon_to_pixel_in_canvas(maxy, minx, zoom, tx_min, ty_min)
    right, bottom = latlon_to_pixel_in_canvas(miny, maxx, zoom, tx_min, ty_min)
    pad = 20
    crop_box = (max(0, left-pad), max(0, top-pad),
                min(canvas.width, right+pad), min(canvas.height, bottom+pad))
    return canvas.crop(crop_box), tx_min, ty_min, crop_box

def draw_polygon_overlay(img, geom, zoom, origin_tx, origin_ty, crop_box):
    overlay = Image.new('RGBA', img.size, (0, 0, 0, 0))
    draw = ImageDraw.Draw(overlay)
    crop_left, crop_top = crop_box[0], crop_box[1]
    def to_px(lon, lat):
        px, py = latlon_to_pixel_in_canvas(lat, lon, zoom, origin_tx, origin_ty)
        return px - crop_left, py - crop_top
    rings = []
    if geom.geom_type == 'Polygon':
        rings = [geom.exterior] + list(geom.interiors)
    elif geom.geom_type == 'MultiPolygon':
        for poly in geom.geoms:
            rings.append(poly.exterior)
            rings.extend(poly.interiors)
    for ring in rings:
        pixels = [to_px(lon, lat) for lon, lat in ring.coords]
        if len(pixels) >= 3:
            draw.polygon(pixels, fill=(0, 200, 0, 55))
            draw.line(pixels + [pixels[0]], fill=(255, 220, 0, 230), width=2)
    return Image.alpha_composite(img.convert('RGBA'), overlay).convert('RGB')

def calc_auto_zoom(bounds, max_zoom=18):
    for z in range(max_zoom, 1, -1):
        tx0, ty0 = latlon_to_tile_xy(bounds[3], bounds[0], z)
        tx1, ty1 = latlon_to_tile_xy(bounds[1], bounds[2], z)
        if max(tx1 - tx0 + 1, ty1 - ty0 + 1) <= 4:
            return z
    return 12

def polygon_uid(geom, idx):
    return hashlib.md5(f'{idx}_{geom.wkt[:200]}'.encode()).hexdigest()[:12]

print('関数定義完了')


In [ ]:
# ===== セル6: 接続テスト（タイル1枚だけ取得・数秒で完了） =====
# 広い範囲をzoom=18で取得すると数百枚になるため、1枚だけで確認する
session = requests.Session()
session.headers.update({'User-Agent': 'farmland-detector/1.0 (research)'})

test_lat, test_lon = 36.10, 140.08  # つくば市付近
tx, ty = latlon_to_tile_xy(test_lat, test_lon, zoom=18)
test_tile = fetch_tile(tx, ty, zoom=18, session=session)

plt.figure(figsize=(5, 5))
plt.imshow(test_tile)
plt.title(f'接続テスト OK（zoom=18, 1タイル=256x256px）\n国土地理院 全国最新写真')
plt.axis('off')
plt.show()
print(f'接続OK - タイル座標: z=18, x={tx}, y={ty}')
print('※ 実際の農地画像取得（セル8）ではポリゴン範囲の複数タイルを自動結合します')

In [ ]:
# ===== セル7: GeoJSON読み込み・分布確認 =====
gdf = gpd.read_file(GEOJSON_PATH)
if gdf.crs and gdf.crs.to_epsg() != 4326:
    gdf = gdf.to_crs(epsg=4326)
gdf = gdf[gdf.geometry.geom_type.isin(['Polygon', 'MultiPolygon'])].reset_index(drop=True)
if MAX_POLYGONS:
    gdf = gdf.head(MAX_POLYGONS)

print(f'処理対象ポリゴン数: {len(gdf)}')
gdf.plot(figsize=(10, 8), color='green', alpha=0.3, edgecolor='black', linewidth=0.3)
plt.title('農地ポリゴン分布')
plt.show()

In [ ]:
# ===== セル8: 画像取得メインループ（並列化 + 小ポリゴン対応版） =====
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
from requests.adapters import HTTPAdapter
from tqdm import tqdm as tqdm_cli

WORKERS  = 4      # 同時並列数。GSIサーバー負荷軽減のため4以上に上げないこと
MIN_SPAN = 0.0003 # これ未満のポリゴンは中心から拡張して取得（約30m）
MAX_SPAN = 0.5    # これ以上はスキップ（約50km）

# ---------- ヘルパー ----------

def make_session():
    s = requests.Session()
    s.headers.update({'User-Agent': 'farmland-detector/1.0 (research)'})
    adapter = HTTPAdapter(pool_connections=WORKERS * 2, pool_maxsize=WORKERS * 4)
    s.mount('https://', adapter)
    return s

def ensure_min_bounds(bounds, min_span=MIN_SPAN):
    """ポリゴンが小さすぎる場合、中心から min_span に広げた bounds を返す。"""
    minx, miny, maxx, maxy = bounds
    cx, cy = (minx + maxx) / 2, (miny + maxy) / 2
    if max(maxx - minx, maxy - miny) < min_span:
        half = min_span / 2
        return (cx - half, cy - half, cx + half, cy + half)
    return bounds

def fetch_region_image_parallel(bounds, zoom, margin=1, session=None):
    """タイルを並列取得してキャンバスに結合・クロップして返す。"""
    minx, miny, maxx, maxy = bounds
    if session is None:
        session = make_session()
    tx_min, ty_min = latlon_to_tile_xy(maxy, minx, zoom)
    tx_max, ty_max = latlon_to_tile_xy(miny, maxx, zoom)
    tx_min -= margin; ty_min -= margin
    tx_max += margin; ty_max += margin
    n_cols = tx_max - tx_min + 1
    n_rows = ty_max - ty_min + 1
    canvas = Image.new('RGB', (n_cols * TILE_SIZE, n_rows * TILE_SIZE))
    tile_jobs = [
        (r, c, ty, tx)
        for r, ty in enumerate(range(ty_min, ty_max + 1))
        for c, tx in enumerate(range(tx_min, tx_max + 1))
    ]
    results = {}
    with ThreadPoolExecutor(max_workers=min(len(tile_jobs), WORKERS * 2)) as ex:
        futures = {ex.submit(fetch_tile, tx, ty, zoom, session): (r, c)
                   for r, c, ty, tx in tile_jobs}
        for fut in as_completed(futures):
            r, c = futures[fut]
            results[(r, c)] = fut.result()
    for (r, c), tile in results.items():
        if tile is not None:
            canvas.paste(tile, (c * TILE_SIZE, r * TILE_SIZE))
    left,  top    = latlon_to_pixel_in_canvas(maxy, minx, zoom, tx_min, ty_min)
    right, bottom = latlon_to_pixel_in_canvas(miny, maxx, zoom, tx_min, ty_min)
    pad = 20
    crop_box = (max(0, left - pad), max(0, top - pad),
                min(canvas.width, right + pad), min(canvas.height, bottom + pad))
    return canvas.crop(crop_box), tx_min, ty_min, crop_box

# ---------- 1ポリゴン処理 ----------

_write_lock = threading.Lock()

def process_polygon(idx, row, output_dir, session):
    """1ポリゴンを処理して (status, data) を返す。"""
    geom     = row.geometry
    uid      = polygon_uid(geom, idx)
    out_path = output_dir / f'{uid}.jpg'

    if out_path.exists():
        return 'exists', None

    bounds = geom.bounds
    span   = max(bounds[2] - bounds[0], bounds[3] - bounds[1])

    if span > MAX_SPAN:
        return 'toobig', None

    fetch_bounds = ensure_min_bounds(bounds)  # 小さいポリゴンは拡張

    # ポリゴン中心でズームレベルの実在確認（zoom=18が404の地域に対応）
    cx = (fetch_bounds[0] + fetch_bounds[2]) / 2
    cy = (fetch_bounds[1] + fetch_bounds[3]) / 2
    available_zoom = find_available_zoom(cy, cx, max_zoom=ZOOM, session=session)
    zoom = calc_auto_zoom(fetch_bounds, max_zoom=available_zoom)

    try:
        img, tx0, ty0, cbox = fetch_region_image_parallel(
            fetch_bounds, zoom=zoom, margin=MARGIN_TILES, session=session)
        img = draw_polygon_overlay(img, geom, zoom, tx0, ty0, cbox)
        img = img.resize((OUT_SIZE, OUT_SIZE), Image.LANCZOS)
        img.save(out_path, 'JPEG', quality=92)
        return 'success', {'uid': uid, 'path': str(out_path), 'zoom': zoom,
                           'span': round(span, 6), 'expanded': span < MIN_SPAN}
    except Exception as e:
        return 'error', str(e)

# ---------- メインループ ----------

output_dir   = Path(OUTPUT_DIR)
session_pool = make_session()

success = skip_exists = skip_toobig = error = 0
meta_records = []

pbar = tqdm_cli(total=len(gdf), desc='画像取得', unit='件', dynamic_ncols=True, leave=True)

with ThreadPoolExecutor(max_workers=WORKERS) as executor:
    futures = {
        executor.submit(process_polygon, idx, row, output_dir, session_pool): idx
        for idx, row in gdf.iterrows()
    }
    for fut in as_completed(futures):
        idx = futures[fut]
        status, data = fut.result()
        if status == 'success':
            success += 1
            with _write_lock:
                meta_records.append(data)
        elif status == 'exists':
            skip_exists += 1
        elif status == 'toobig':
            skip_toobig += 1
        else:
            error += 1
            tqdm_cli.write(f'  [error] idx={idx}: {data}')
        pbar.update(1)
        pbar.set_postfix(成功=success, 既存=skip_exists, 大きすぎ=skip_toobig, エラー=error)

pbar.close()

# metadata.csv に追記（既存データとマージ）
meta_path = output_dir / 'metadata.csv'
if meta_path.exists() and meta_records:
    existing = pd.read_csv(meta_path)
    merged = pd.concat([existing, pd.DataFrame(meta_records)]).drop_duplicates('uid')
    merged.to_csv(meta_path, index=False)
elif meta_records:
    pd.DataFrame(meta_records).to_csv(meta_path, index=False)

print(f'\n--- 完了 ---')
print(f'  成功          : {success} 件')
print(f'  既存スキップ   : {skip_exists} 件')
print(f'  大きすぎスキップ: {skip_toobig} 件')
print(f'  エラー        : {error} 件')
print(f'  合計保存済み   : {len(list(output_dir.glob("*.jpg")))} 枚')


In [ ]:
# ===== セル9: 取得画像のサムネイル確認（先頭12枚） =====
images  = sorted(output_dir.glob('*.jpg'))[:12]
meta_df = pd.read_csv(output_dir / 'metadata.csv') if (output_dir / 'metadata.csv').exists() else pd.DataFrame()
fig, axes = plt.subplots(3, 4, figsize=(16, 12))
for ax, p in zip(axes.flat, images):
    ax.imshow(Image.open(p))
    row = meta_df[meta_df.uid == p.stem] if len(meta_df) else pd.DataFrame()
    z = int(row.zoom.values[0]) if len(row) else '?'
    ax.set_title(f'{p.stem[:8]}\nzoom={z}', fontsize=7)
    ax.axis('off')
for ax in axes.flat[len(images):]:
    ax.axis('off')
plt.suptitle('取得した農地画像（先頭12枚） - 国土地理院 全国最新写真')
plt.tight_layout()
plt.show()

In [ ]:
# ===== セル10: ラベリング状況確認 =====
# 画像取得後、data/unlabeled/ の画像を目視で確認し
# data/farmland/ → 正常な農地
# data/problem/  → 建物・道路が1/3以上の問題エリア
# に手動で振り分けてからこのセルを実行してください（各クラス50枚以上推奨）

for cls in ['farmland', 'problem', 'unlabeled', 'review']:
    n = len(list(Path(f'data/{cls}').glob('*.jpg'))) if Path(f'data/{cls}').exists() else 0
    print(f'  {cls:12s}: {n} 枚')

In [ ]:
# ===== セル11: モデル・学習関数の定義 =====
DATA_DIR   = 'data'
MODEL_OUT  = 'models/model_v1.pth'
EPOCHS     = 20
BATCH_SIZE = 16   # メモリが足りなければ 8 に下げる
LR         = 1e-4
VAL_RATIO  = 0.15

def build_transforms(train=True):
    if train:
        return T.Compose([
            T.Resize((224, 224)),
            T.RandomHorizontalFlip(),
            T.RandomVerticalFlip(),
            T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
            T.RandomRotation(15),
            T.ToTensor(),
            T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ])
    return T.Compose([
        T.Resize((224, 224)),
        T.ToTensor(),
        T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])

def build_model(num_classes=2):
    model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, num_classes)
    return model

def run_epoch(model, loader, criterion, optimizer=None):
    training = optimizer is not None
    model.train() if training else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            if training:
                optimizer.zero_grad()
            out = model(imgs)
            loss = criterion(out, labels)
            if training:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * imgs.size(0)
            correct += (out.argmax(1) == labels).sum().item()
            total += imgs.size(0)
    return total_loss / total, correct / total

print('モデル関数定義完了')

In [ ]:
# ===== セル12: 学習実行 =====
full_ds = datasets.ImageFolder(DATA_DIR, transform=build_transforms(train=True))
print(f'クラス: {full_ds.class_to_idx}')
print(f'総サンプル数: {len(full_ds)}')

n_val   = max(1, int(len(full_ds) * VAL_RATIO))
n_train = len(full_ds) - n_val
train_ds, val_ds = random_split(full_ds, [n_train, n_val], generator=torch.Generator().manual_seed(42))
val_ds.dataset = datasets.ImageFolder(DATA_DIR, transform=build_transforms(train=False))

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
print(f'train={n_train}枚, val={n_val}枚')

model = build_model().to(DEVICE)
counts = [0] * 2
for _, label in full_ds.samples:
    counts[label] += 1
class_weights = torch.tensor([1.0 / c for c in counts], dtype=torch.float).to(DEVICE)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

history, best_val_acc = [], 0.0
Path(MODEL_OUT).parent.mkdir(parents=True, exist_ok=True)

for epoch in tqdm(range(1, EPOCHS + 1), desc='学習'):
    train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer)
    val_loss,   val_acc   = run_epoch(model, val_loader,   criterion)
    scheduler.step()
    history.append({'epoch': epoch, 'train_loss': train_loss, 'train_acc': train_acc,
                    'val_loss': val_loss, 'val_acc': val_acc})
    tqdm.write(f'Epoch {epoch:03d} | train acc={train_acc:.3f} loss={train_loss:.4f} | val acc={val_acc:.3f} loss={val_loss:.4f}')
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({'epoch': epoch, 'model': model.state_dict(),
                    'class_to_idx': full_ds.class_to_idx, 'val_acc': val_acc}, MODEL_OUT)
        tqdm.write(f'  → モデル保存 (val_acc={val_acc:.4f})')

print(f'\n学習完了。最良 val_acc={best_val_acc:.4f}')

In [ ]:
# ===== セル13: 学習曲線グラフ =====
df_hist = pd.DataFrame(history)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(df_hist.epoch, df_hist.train_loss, label='train')
ax1.plot(df_hist.epoch, df_hist.val_loss,   label='val')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.legend(); ax1.set_title('Loss')
ax2.plot(df_hist.epoch, df_hist.train_acc, label='train')
ax2.plot(df_hist.epoch, df_hist.val_acc,   label='val')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy'); ax2.legend(); ax2.set_title('Accuracy')
ax2.set_ylim(0, 1)
plt.tight_layout()
plt.savefig('logs/training_curve.png', dpi=150)
plt.show()

In [ ]:
# ===== セル14: 推論設定 =====
PREDICT_MODEL = 'models/model_v1.pth'  # 使用するモデル（継続学習後は model_v2.pth など）
PREDICT_INPUT = 'data/unlabeled'       # 推論対象フォルダ
THRESHOLD     = 0.85                   # この確信度未満は review/ へ
DRY_RUN       = True                   # True=移動しない（結果確認用）。実際に移動するには False に

In [ ]:
# ===== セル15: 推論実行 =====
state = torch.load(PREDICT_MODEL, map_location=DEVICE)
pred_model = build_model().to(DEVICE)
pred_model.load_state_dict(state['model'])
pred_model.eval()
idx_to_class = {v: k for k, v in state['class_to_idx'].items()}
print(f'モデル読み込み完了 (val_acc={state["val_acc"]:.4f})')

infer_transform = build_transforms(train=False)
input_dir = Path(PREDICT_INPUT)
out_dirs  = {
    'farmland': input_dir.parent / 'farmland',
    'problem':  input_dir.parent / 'problem',
    'review':   input_dir.parent / 'review',
}
if not DRY_RUN:
    for d in out_dirs.values():
        d.mkdir(parents=True, exist_ok=True)

images, records = sorted(input_dir.glob('*.jpg')), []
with torch.no_grad():
    for img_path in tqdm(images, desc='推論'):
        try:
            tensor = infer_transform(Image.open(img_path).convert('RGB')).unsqueeze(0).to(DEVICE)
            probs  = F.softmax(pred_model(tensor), dim=1)[0]
            pred_idx   = probs.argmax().item()
            confidence = probs[pred_idx].item()
            pred_class = idx_to_class[pred_idx]
            dest = pred_class if confidence >= THRESHOLD else 'review'
            records.append({'file': img_path.name, 'pred': pred_class,
                            'confidence': round(confidence, 4), 'dest': dest})
            if not DRY_RUN:
                shutil.move(str(img_path), out_dirs[dest] / img_path.name)
        except Exception as e:
            tqdm.write(f'  [skip] {img_path.name}: {e}')

df_pred = pd.DataFrame(records)
df_pred.to_csv('data/predict_report.csv', index=False)
print('\n--- 推論結果 ---')
print(df_pred.dest.value_counts().to_string())
print(f'\n平均確信度: {df_pred.confidence.mean():.4f}')
if DRY_RUN:
    print('\n※ DRY_RUN=True のため移動していません。False にして再実行してください。')

In [ ]:
# ===== セル16: 確信度分布グラフ + review画像サムネイル =====
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df_pred.confidence, bins=30, edgecolor='black')
axes[0].axvline(THRESHOLD, color='red', linestyle='--', label=f'threshold={THRESHOLD}')
axes[0].set_xlabel('確信度'); axes[0].set_ylabel('件数')
axes[0].set_title('確信度分布'); axes[0].legend()
df_pred.dest.value_counts().plot(kind='bar', ax=axes[1])
axes[1].set_title('振り分け結果'); axes[1].set_xlabel('')
plt.tight_layout()
plt.show()

review_images = sorted(Path('data/review').glob('*.jpg'))[:16]
if review_images:
    cols = 4
    rows = (len(review_images) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(16, rows * 4))
    for ax, p in zip(axes.flat, review_images):
        row = df_pred[df_pred.file == p.name]
        conf = row.confidence.values[0] if len(row) else 0
        pred = row.pred.values[0] if len(row) else '?'
        ax.imshow(Image.open(p))
        ax.set_title(f'pred={pred}\nconf={conf:.2f}', fontsize=8)
        ax.axis('off')
    for ax in axes.flat[len(review_images):]:
        ax.axis('off')
    plt.suptitle('要確認画像（review/）→ farmland/ or problem/ に手動で移動してください')
    plt.tight_layout()
    plt.show()
else:
    print('review/ に画像がありません')

In [ ]:
# ===== セル17: 継続学習（Active Learning） =====
# 実行前に: data/review/ の画像を farmland/ または problem/ に手動で移動してください

BASE_MODEL    = 'models/model_v1.pth'  # 起点モデル（継続学習のたびに更新）
NEW_MODEL_OUT = 'models/model_v2.pth'  # 新バージョンの保存先
RETRAIN_EPOCHS = 10
RETRAIN_LR     = 5e-5  # 継続学習は小さめの学習率

# データ状況チェック
review_n   = len(list(Path('data/review').glob('*.jpg')))
farmland_n = len(list(Path('data/farmland').glob('*.jpg')))
problem_n  = len(list(Path('data/problem').glob('*.jpg')))
print(f'farmland: {farmland_n} 枚 / problem: {problem_n} 枚 / review: {review_n} 枚')
if review_n > 0:
    print('⚠️  review/ に画像が残っています。移動してから再実行してください。')
else:
    retrain_ds = datasets.ImageFolder(DATA_DIR, transform=build_transforms(train=True))
    n_val_r   = max(1, int(len(retrain_ds) * VAL_RATIO))
    n_train_r = len(retrain_ds) - n_val_r
    train_ds_r, val_ds_r = random_split(retrain_ds, [n_train_r, n_val_r], generator=torch.Generator().manual_seed(42))
    val_ds_r.dataset = datasets.ImageFolder(DATA_DIR, transform=build_transforms(train=False))
    train_loader_r = DataLoader(train_ds_r, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
    val_loader_r   = DataLoader(val_ds_r,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    retrain_model = build_model().to(DEVICE)
    retrain_state = torch.load(BASE_MODEL, map_location=DEVICE)
    retrain_model.load_state_dict(retrain_state['model'])
    print(f'起点モデル val_acc={retrain_state["val_acc"]:.4f}')

    counts_r = [0] * 2
    for _, label in retrain_ds.samples:
        counts_r[label] += 1
    criterion_r = nn.CrossEntropyLoss(
        weight=torch.tensor([1.0/c for c in counts_r], dtype=torch.float).to(DEVICE))
    optimizer_r = torch.optim.AdamW(retrain_model.parameters(), lr=RETRAIN_LR, weight_decay=1e-4)
    scheduler_r = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_r, T_max=RETRAIN_EPOCHS)

    best_r = 0.0
    Path(NEW_MODEL_OUT).parent.mkdir(parents=True, exist_ok=True)
    for epoch in tqdm(range(1, RETRAIN_EPOCHS + 1), desc='継続学習'):
        _, train_acc_r = run_epoch(retrain_model, train_loader_r, criterion_r, optimizer_r)
        _, val_acc_r   = run_epoch(retrain_model, val_loader_r,   criterion_r)
        scheduler_r.step()
        tqdm.write(f'Epoch {epoch:03d} | train={train_acc_r:.3f} | val={val_acc_r:.3f}')
        if val_acc_r > best_r:
            best_r = val_acc_r
            torch.save({'epoch': epoch, 'model': retrain_model.state_dict(),
                        'class_to_idx': retrain_ds.class_to_idx, 'val_acc': val_acc_r}, NEW_MODEL_OUT)
            tqdm.write(f'  → 保存: {NEW_MODEL_OUT}')

    print(f'\n継続学習完了: {retrain_state["val_acc"]:.4f} → {best_r:.4f}')
    print('次回: PREDICT_MODEL と BASE_MODEL を', NEW_MODEL_OUT, 'に変更して推論→仕分け→継続学習を繰り返す')